# 17 - Ablation: frozen features + linear head under notebook 16's protocol

The frozen half of a 2 x 2 comparison (frozen vs fine-tuned image model x regex vs LLM
soft labels). It reuses notebook 14's frozen ResNet-18 features (`features_v1.npz`) and
trains a linear 12-target head **exactly under notebook 16's protocol**:

* trains on folds 1-4 without the 58 expert-labelled studies or any study sharing a report
  with them; picks the epoch on fold 0's weakly labelled studies; scores the 58 once;
* unmasked soft binary cross-entropy against the label matrix, as in 16;
* run twice: on notebook 11's regex scores and on 11c's soft labels.

Unlike 14 the expert labels are never trained on, so these numbers are not 14's 0.663.
CPU, internet off. Inputs: 11, 11c, 12, 14. Outputs: `ablation_frozen_v1.json`,
`gold_preds_regex_v1.csv`, `gold_preds_soft_v1.csv`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# The competition mount path has differed between Kaggle runtimes; take the first that has train.csv.
_SLUG = "rsna-knee-abnormality-detection"
_roots = [Path("/kaggle/input/competitions") / _SLUG, Path("/kaggle/input") / _SLUG,
          *sorted(Path("/kaggle/input").glob("*/" + _SLUG)), *sorted(Path("/kaggle/input").glob("*"))]
ROOT = next((r for r in _roots if (r / "train.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("competition train.csv not found; /kaggle/input has: "
                            + str(sorted(str(p) for p in Path("/kaggle/input").glob("*/*"))[:50]))
print("competition data:", ROOT)
OUT = Path("/kaggle/working")
TARGET_COLUMNS = [
    "ACL", "MCL",
    "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]
SEED = 42
from pathlib import Path


def find_input(name):
    """Locate one attached upstream file by name.

    Only notebook outputs and datasets are searched: walking the competition
    mount would visit millions of DICOM files.
    """
    hits = []
    for base in (Path("/kaggle/input/notebooks"), Path("/kaggle/input/datasets"),
                 Path("/kaggle/input")):
        if base.is_dir():
            for child in base.iterdir():
                if (child.is_dir()
                        and child.name not in ("competitions", "notebooks", "datasets")
                        and not (child / "train_series").exists()
                        and not (child / "test_series").exists()):
                    hits += list(child.rglob(name))
    hits = sorted(set(hits))
    if len(hits) != 1:
        raise FileNotFoundError(
            f"expected exactly one attached '{name}', found {len(hits)}: {hits[:5]}. "
            "Attach the upstream notebook output via Add Input."
        )
    return hits[0]

from sklearn.metrics import roc_auc_score


def safe_auc(y, p):
    """AUC, or NaN when only one class is present (the metric is undefined)."""
    y = np.asarray(y)
    return float(roc_auc_score(y, p)) if np.unique(y).size == 2 else np.nan


def auc_table(gold, scores, n_boot=2000, seed=0):
    """Per-target AUC with a percentile bootstrap interval over studies."""
    rng = np.random.default_rng(seed)
    rows = []
    for t in TARGET_COLUMNS:
        y, p = gold[t].to_numpy(), scores[t].to_numpy()
        boots = []
        for _ in range(n_boot):
            i = rng.integers(0, len(y), len(y))
            a = safe_auc(y[i], p[i])
            if np.isfinite(a):
                boots.append(a)
        rows.append({"target": t, "auc": safe_auc(y, p),
                     "ci_low": np.percentile(boots, 2.5) if boots else np.nan,
                     "ci_high": np.percentile(boots, 97.5) if boots else np.nan,
                     "n_pos": int(y.sum()), "n_neg": int((1 - y).sum())})
    return pd.DataFrame(rows)


def macro_auc(table):
    """Challenge metric: mean of per-target AUCs; undefined targets excluded and counted."""
    return float(table["auc"].mean(skipna=True)), int(table["auc"].notna().sum())

import json
import random

import torch
from torch import nn

EPOCHS, LR, WEIGHT_DECAY, VAL_FOLD = 400, 1e-2, 1e-3, 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

weak = pd.read_csv(find_input("weak_labels_v1.csv"), dtype={"StudyInstanceUID": str})
soft = pd.read_csv(find_input("soft_labels_v1.csv"), dtype={"StudyInstanceUID": str})
folds = pd.read_csv(find_input("folds_v1.csv"), dtype={"StudyInstanceUID": str})
with np.load(find_input("features_v1.npz"), allow_pickle=True) as f:
    feats = pd.DataFrame(f["X"], index=f["studies"].astype(str))

data = (folds[["StudyInstanceUID", "fold", "group", "is_gold"]]
        .merge(weak[["StudyInstanceUID"] + [t + s for t in TARGET_COLUMNS for s in ("__weak", "__gold")]],
               on="StudyInstanceUID", validate="one_to_one")
        .merge(soft[["StudyInstanceUID"] + TARGET_COLUMNS], on="StudyInstanceUID", validate="one_to_one"))
assert data["StudyInstanceUID"].isin(feats.index).all(), "every study needs a feature row"
# Split: same rule as notebook 16.
gold_groups = set(data.loc[data["is_gold"].eq(1), "group"])
near_gold = data["group"].isin(gold_groups) & data["is_gold"].eq(0)
is_gold = data["is_gold"].eq(1)
train_df = data[~is_gold & ~near_gold & data["fold"].ne(VAL_FOLD)]
val_df = data[~is_gold & ~near_gold & data["fold"].eq(VAL_FOLD)]
gold_df = data[is_gold]
assert not set(train_df["StudyInstanceUID"]) & set(gold_df["StudyInstanceUID"])
print("train", len(train_df), "| epoch selection", len(val_df), "| expert-labelled (final only)", len(gold_df))

X = {k: feats.loc[d["StudyInstanceUID"]].to_numpy(np.float32) for k, d in
     (("train", train_df), ("val", val_df), ("gold", gold_df))}
mean, std = X["train"].mean(0, keepdims=True), X["train"].std(0, keepdims=True)
std = np.where(std < 1e-6, 1.0, std)
Z = {k: torch.from_numpy((v - mean) / std).float() for k, v in X.items()}
gold_y = gold_df[[t + "__gold" for t in TARGET_COLUMNS]].set_axis(TARGET_COLUMNS, axis=1).astype(int)

In [ ]:
def fit_head(y_train, y_val):
    """Full-batch AdamW on soft BCE; keeps the epoch with the lowest fold-0 loss."""
    torch.manual_seed(SEED)
    yt, yv = torch.from_numpy(y_train), torch.from_numpy(y_val)
    head = nn.Linear(Z["train"].shape[1], 12)
    prior = y_train.mean(0).clip(1e-3, 1 - 1e-3)
    with torch.no_grad():
        head.weight.zero_()
        head.bias.copy_(torch.from_numpy(np.log(prior / (1 - prior))).float())
    opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    bce = nn.functional.binary_cross_entropy_with_logits
    best = (float("inf"), 0, None)
    for epoch in range(1, EPOCHS + 1):
        opt.zero_grad()
        bce(head(Z["train"]), yt).backward()
        opt.step()
        with torch.no_grad():
            v = bce(head(Z["val"]), yv).item()
        if v < best[0]:
            best = (v, epoch, {k: t.clone() for k, t in head.state_dict().items()})
    head.load_state_dict(best[2])
    return head.eval(), best[1], best[0]


results = {}
for name, cols in (("regex", [t + "__weak" for t in TARGET_COLUMNS]), ("soft", TARGET_COLUMNS)):
    head, epoch, val_bce = fit_head(train_df[cols].to_numpy(np.float32), val_df[cols].to_numpy(np.float32))
    with torch.no_grad():
        pred = pd.DataFrame(torch.sigmoid(head(Z["gold"])).numpy(), columns=TARGET_COLUMNS, index=gold_df.index)
    table = auc_table(gold_y, pred)
    macro, n_eval = macro_auc(table)
    results[name] = {"macro_auc": macro, "evaluable_targets": n_eval, "epoch": epoch, "val_bce": val_bce,
                     "per_target": table.to_dict("records")}
    pd.concat([gold_df[["StudyInstanceUID"]], pred.add_suffix("__pred")], axis=1).to_csv(
        OUT / f"gold_preds_{name}_v1.csv", index=False)
    print(f"frozen + {name:5s}: expert-labelled macro AUC {macro:.4f} over {n_eval}/12 | "
          f"epoch {epoch} | fold-0 BCE {val_bce:.4f}")
(OUT / "ablation_frozen_v1.json").write_text(json.dumps(
    {"protocol": "notebook 16 split; no expert labels in training", "n_train": len(train_df),
     "n_val": len(val_df), **results}, indent=2, default=float))